# Adaptive A\* — independent re-implementation and reproduction (Kaggle runner)

Re-implementation of H. Cao *et al.*, "An Adaptive A\* Algorithm for Mobile Robots Global Path Planning," *Electronics* 15(13), 2807, 2026.

This notebook runs the **whole repository** end to end, on the paper's own maps (digitised from its Fig. 5):
1. unit tests;
2. digitising the paper's Fig. 5 into the three grid maps;
3. reproduction of Tables 1–5 against the published numbers, plus the 150-map robustness study;
4. the paired timing study;
5. comparison with the paper's result figures (Figs. 8–9) and the admissibility check;
6. all figures.

It then shows the results inline and packs them into a zip you can download.

**CPU only.** No GPU or accelerator is needed. A full run takes about 5–10 minutes.

### How to get the code into the notebook (choose one)
- **A. Kaggle Dataset (no GitHub, works offline).** Upload `adaptive-astar-reimplementation.zip` as a Kaggle Dataset, then *Add Input* → that dataset. The next cell finds it automatically under `/kaggle/input`.
- **B. GitHub clone (needs Internet).** Turn on *Settings → Internet*. `REPO_URL` and `BRANCH` are set below.

In [ ]:
# ---- configuration -------------------------------------------------------
REPO_URL = "https://github.com/Mariam1543/aqqal123.git"   # used only for option B
BRANCH = "claude/relaxed-johnson-4f4051"

QUICK = False          # True = smoke run (about 1 min); False = full run as in the report
TIMING_REPEATS = 3 if QUICK else 30
ROBUST_MAPS = 5 if QUICK else 50
TIMING_PAIRS = 30 if QUICK else 300

In [ ]:
# ---- locate the code and copy it to a writable directory -------------------
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
PROJECT = WORK / "adaptive-astar-reimplementation"

def find_in_inputs():
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for marker in root.rglob("adaptive_astar/__init__.py"):          # unpacked folder
        return marker.parents[2]
    for z in root.rglob("*.zip"):                                     # zipped upload
        with zipfile.ZipFile(z) as zf:
            zf.extractall(WORK / "_unzipped")
        for marker in (WORK / "_unzipped").rglob("adaptive_astar/__init__.py"):
            return marker.parents[2]
    return None

if not (PROJECT / "src" / "adaptive_astar").exists():
    src = find_in_inputs()
    if src is not None:                                               # option A
        shutil.copytree(src, PROJECT, dirs_exist_ok=True)
        print("copied code from Kaggle input:", src)
    else:                                                             # option B
        url = REPO_URL
        token = None
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        if token:
            url = url.replace("https://", f"https://{token}@")
        clone_dir = WORK / "_repo"
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(clone_dir)], check=True)
        shutil.copytree(clone_dir / "adaptive-astar-reimplementation", PROJECT, dirs_exist_ok=True)
        print("cloned", REPO_URL, "@", BRANCH)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
print("project directory:", PROJECT)
print(sorted(p.name for p in PROJECT.iterdir()))

In [ ]:
# ---- environment check (all packages are pre-installed on Kaggle) --------
import platform, numpy, pandas, scipy, matplotlib
print("python", platform.python_version(), "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
print("CPUs:", os.cpu_count())

In [ ]:
def run(*args):
    """Run a project script, streaming its output; stop the notebook on failure."""
    print("$ python", " ".join(args), flush=True)
    proc = subprocess.run([sys.executable, *args], cwd=PROJECT, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-6000:])
    if proc.returncode != 0:
        raise RuntimeError(f"{args[0]} failed with exit code {proc.returncode}")

run("-m", "unittest", "discover", "-s", "tests")

## Step 1 — Digitise the paper's maps from its Fig. 5
Rebuilds `maps/paper_map{1,2,3}.csv` from `data/paper_figure5_maps.png` and draws the side-by-side check.

In [ ]:
run("scripts/digitize_figure5.py")

## Step 2 — Reproduce Tables 1–5 and the robustness study (paper maps)
Main results use Eqs. 5–7 exactly as printed. The V4 rows are a sensitivity check only.

In [ ]:
run("scripts/run_experiments.py", "--map-set", "paper", "--repeats", str(TIMING_REPEATS), "--robust-maps", str(ROBUST_MAPS))

## Step 3 — Paired timing study, figure comparison, admissibility, figures

In [ ]:
run("scripts/timing_study.py", "--map-set", "paper", "--pairs", str(TIMING_PAIRS))
run("scripts/compare_paper_figures.py")
run("scripts/admissibility.py")
run("scripts/make_figures.py", "--map-set", "paper")

## Results

In [ ]:
from IPython.display import Markdown, Image, display

RES = PROJECT / "results" / "paper_maps"
FIG = PROJECT / "figures" / "paper_maps"
comparison = (RES / "comparison.md").read_text()
display(Markdown(comparison.split("# Published vs reproduced results")[0]))   # headline table (Metrics 1-4)
display(Markdown((RES / "timing_study.md").read_text()))

In [ ]:
# Full paper-vs-ours tables (Tables 1-5)
display(Markdown(comparison))

In [ ]:
# Robustness, comparison with the paper's figures, admissibility
for name in ("robustness.md", "figure_agreement.md", "admissibility.md"):
    display(Markdown((RES / name).read_text()))

In [ ]:
display(Image(filename=str(PROJECT / "figures" / "fig_digitised_maps.png")))
for name in ["fig_paper_vs_ours_panels", "fig_relative_change", "fig_robustness", "fig_weight_heuristic",
             "fig_ablation_30x30", "fig_ablation_60x60",
             "fig_algorithms_30x30", "fig_algorithms_60x60", "fig_algorithms_100x100",
             "fig_start_goal"]:
    print(name)
    display(Image(filename=str(FIG / f"{name}.png")))

In [ ]:
# ---- package outputs for download (Output tab -> adaptive_astar_outputs.zip) --
out = WORK / "adaptive_astar_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("results/paper_maps", "figures", "maps"):
        for f in sorted((PROJECT / folder).rglob("*")):
            if f.is_file():
                zf.write(f, f.relative_to(PROJECT))
print("download:", out, f"({out.stat().st_size / 1e6:.1f} MB)")